# Recommendation System (Content-Based Filtering)
**Repositori**: Machine Learning
**Topik**: Implementasi content-based recommendation system untuk rekomendasi pelanggan
**Dataset**: E-commerce Customer Behavior.csv
---
**Pendahuluan**: Recommendation system membantu pengguna menemukan item yang relevan. Content-based filtering merekomendasikan item berdasarkan kemiripan fitur dengan item yang disukai pengguna sebelumnya.


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Load & EDA


In [ ]:
df = pd.read_csv('../../data/E-commerce Customer Behavior.csv')
print('Shape:', df.shape)
print(df.head())
print(df['Satisfaction Level'].value_counts())
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='Satisfaction Level', order=df['Satisfaction Level'].value_counts().index)
plt.title('Distribusi Satisfaction Level')
plt.subplot(1, 2, 2)
sns.histplot(data=df, x='Total Spend', hue='Satisfaction Level', bins=20)
plt.title('Distribusi Total Spend per Satisfaction')
plt.tight_layout()
plt.show()


## 3. Data Preparation


In [ ]:
df_clean = df.drop(['Customer ID', 'City'], axis=1)
cat_cols = ['Gender', 'Membership Type', 'Satisfaction Level']
for col in cat_cols:
    df_clean[col] = LabelEncoder().fit_transform(df_clean[col])
customer_features = df_clean.drop('Satisfaction Level', axis=1)
customer_labels = df_clean['Satisfaction Level']
scaler = StandardScaler()
features_scaled = scaler.fit_transform(customer_features)
print('Feature shape:', features_scaled.shape)


## 4. Content-Based Filtering dengan Cosine Similarity


In [ ]:
cosine_sim = cosine_similarity(features_scaled)
print('Similarity matrix shape:', cosine_sim.shape)


## 5. Fungsi Rekomendasi


In [ ]:
def recommend_customers(customer_id, df, sim_matrix, n=5):
    idx = df[df['Customer ID'] == customer_id].index[0]
    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    rec_indices = [i[0] for i in sim_scores]
    return df.iloc[rec_indices][['Customer ID', 'Gender', 'Age', 'Total Spend', 'Satisfaction Level']]

sample_id = df['Customer ID'].iloc[0]
print(f'Customer reference: {sample_id}')
print('Rekomendasi customer serupa:')
print(recommend_customers(sample_id, df, cosine_sim))


## 6. K-NN untuk Rekomendasi


In [ ]:
knn = NearestNeighbors(n_neighbors=6, metric='cosine')
knn.fit(features_scaled)
distances, indices = knn.kneighbors(features_scaled[0:1])
print('K-NN Rekomendasi (cosine distance):')
print(df.iloc[indices[0]][['Customer ID', 'Total Spend', 'Satisfaction Level']])


## 7. Rekomendasi Berdasarkan Segmentasi


In [ ]:
df['SpendSegment'] = pd.qcut(df['Total Spend'], q=3, labels=['Low', 'Medium', 'High'])
segment_profiles = df.groupby('SpendSegment')[['Age', 'Items Purchased', 'Average Rating']].mean()
print('Profil per Spending Segment:')
print(segment_profiles)
print('Contoh: Untuk customer High Spender, rekomendasikan item dengan rating tinggi')
recommended = df[df['SpendSegment'] == 'High'].nlargest(5, 'Average Rating')[['Customer ID', 'Total Spend', 'Average Rating', 'Satisfaction Level']]
print(recommended)


## 8. Kesimpulan
Content-based filtering dengan cosine similarity efektif untuk menemukan customer/profile serupa. K-NN memberikan pendekatan alternatif yang scalable. Pendekatan segmentasi membantu memberikan rekomendasi yang lebih kontekstual berdasarkan profil pengguna.
